In [ ]:
# Configuracao do ambiente (celula de setup do notebook)
import sys, os, warnings
sys.path.insert(0, os.path.abspath(os.path.join('..', '..', 'src')))
warnings.filterwarnings('ignore')
%matplotlib inline

# Capítulo 6 — ESCULPINDO O SINAL: FEATURE ENGINEERING PARA DECISÕES

> "O escultor não cria a estátua; ele remove o mármore excedente para revelar a forma que já estava lá."
> — Aforismo de um Cientista de Dados

A análise exploratória me deu um mapa, mas um mapa é uma representação passiva. Agora eu queria ser um agente ativo na paisagem — construir pontes, abrir atalhos entre a dúvida e o diagnóstico. Isso é a **Engenharia de Features**: transformar dados brutos em preditores mais expressivos.

No OncoClassify 1.0, usei as 30 *features* como me foram dadas, sem perguntar o que a *razão* entre duas delas poderia revelar. Desta vez, eu seria um escultor — criaria novas *features* com base na intuição da EDA e no conhecimento do domínio. Mas, como veremos, esculpir exige honestidade: nem todo corte melhora a estátua.

## Engenharia de Features Para Classificação

Feature engineering é usar conhecimento de domínio para criar novas variáveis que **maximizem a separabilidade entre as classes**. No nosso caso, sabemos da EDA que tamanho e irregularidade de forma importam. Vamos criar três *razões* que capturam esses aspectos de forma mais explícita.

Uma advertência desde já: é fácil se empolgar e criar dezenas de *features*, aumentando o risco de **overfitting** e de redundância. Criaremos poucas, com hipótese clara, e — o passo que o OncoClassify 1.0 pulou — **mediremos se elas de fato ajudam**.

#### Código 6.1: Criando Features Derivadas


In [ ]:
from oncoclassify_utils import X_train, y_train

def adicionar_features_derivadas(X):
    """Cria razões morfológicas motivadas por forma e tamanho do núcleo."""
    X = X.copy()
    # Densidade relativa: raio grande para pouca textura pode indicar massa sólida.
    X["ratio_raio_textura"]     = X["mean radius"] / X["mean texture"]
    # Irregularidade da forma: quanto o contorno se afasta de um círculo.
    X["irregularidade_forma"]   = (X["mean perimeter"] ** 2) / (X["mean area"] + 1e-6)
    # Deformação: concavidade acentuada frente à simetria.
    X["assimetria_concavidade"] = X["worst concavity"] / (X["worst symmetry"] + 1e-6)
    return X

NOVAS = ["ratio_raio_textura", "irregularidade_forma", "assimetria_concavidade"]
X_train_eng = adicionar_features_derivadas(X_train)
print(X_train_eng[NOVAS].head().round(4).to_string())

     ratio_raio_textura  irregularidade_forma  assimetria_concavidade
22               1.0757               14.9152                  1.3510
318              0.4784               14.7583                  1.4702
592              1.0329               14.0614                  1.8175
324              0.8021               13.2902                  0.4386
42               0.7686               14.9102                  1.5507


> **📘 PARA SABER — Por que essas razões?**
>
> Métricas de forma como *compactness* e razões envolvendo simetria e concavidade aparecem na literatura de morfometria celular: tumores malignos tendem a contornos menos uniformes e estruturas mais assimétricas (Wolberg, Street & Mangasarian, 1995). Nossas três *features* traduzem essa intuição clínica em números.

#### Código 6.2: As Novas Features Separam as Classes?


In [ ]:
import seaborn as sns, matplotlib.pyplot as plt
from oncoclassify_utils import color_map, apply_hatches_boxplot

df_eng = X_train_eng.copy()
df_eng["diagnosis_label"] = y_train.map({0: "Maligno", 1: "Benigno"}).values
ordem = ["Benigno", "Maligno"]; pal = [color_map[c] for c in ordem]

for feature in NOVAS:
    fig, ax = plt.subplots(figsize=(9, 5))
    sns.boxplot(data=df_eng, x="diagnosis_label", y=feature, hue="diagnosis_label",
                order=ordem, palette=pal, legend=False, ax=ax)
    apply_hatches_boxplot(ax, ordem)
    ax.set_title(f"Box Plot: {feature} por Diagnóstico"); ax.set_xlabel("Diagnóstico")
    plt.tight_layout(); plt.show()

![ratio_raio_textura](media/figura_06_1.png)

![irregularidade_forma](media/figura_06_2.png)

![assimetria_concavidade](media/figura_06_3.png)

Individualmente, as três carregam sinal — especialmente `assimetria_concavidade`, cujos valores malignos são visivelmente mais altos. Um teste rápido confirma: essa *feature* tem correlação de Spearman de **0,66** com a malignidade. Parece promissor. Mas "prometer isoladamente" não é o mesmo que "ajudar o modelo".

#### Código 6.3: O Teste Honesto — Elas Melhoram o Modelo?


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score

for nome, modelo in [("LogReg", LogisticRegression(solver="liblinear", random_state=42)),
                     ("SVM-RBF", SVC(kernel="rbf", random_state=42))]:
    pipe = make_pipeline(StandardScaler(), modelo)
    acc_30  = cross_val_score(pipe, X_train,     y_train, cv=5).mean()
    acc_eng = cross_val_score(pipe, X_train_eng, y_train, cv=5).mean()
    print(f"{nome:8s}  30 features={acc_30:.4f}   30+engenharia={acc_eng:.4f}   "
          f"(ganho {acc_eng - acc_30:+.4f})")

LogReg    30 features=0.9688   30+engenharia=0.9688   (ganho +0.0000)
SVM-RBF   30 features=0.9750   30+engenharia=0.9708   (ganho -0.0042)


> **⚠️ ATENÇÃO — Mais features não é melhor modelo**
>
> O veredicto é desconfortável e valioso: sobre as 30 *features* originais, as três novas praticamente **não mudam nada** (ganho **exatamente zero** na Regressão Logística, e até **pioram** o SVM em 0,4 ponto). Por quê? Porque a informação delas *já estava* nas originais — `assimetria_concavidade` é forte, mas redundante com `worst concavity` e `worst symmetry`, que já usamos. Adicionar redundância só aumenta o risco de *overfitting* sem trazer sinal novo. Esta é uma das lições mais difíceis do *machine learning*: **a criatividade precisa ser validada por evidência.** Guardamos o aprendizado e seguimos enxutos.

> **📌 CHECKLIST DO CAPÍTULO 6**
>
> ✅ Entende o objetivo da engenharia de *features*: maximizar separabilidade.
>
> ✅ Criou três *features* derivadas com hipótese de domínio clara.
>
> ✅ Viu que uma delas (`assimetria_concavidade`) carrega sinal individual (corr. 0,66).
>
> ✅ **Testou** o efeito real e descobriu que, sobre as 30 originais, o ganho é nulo ou negativo.
>
> **⚠️ CRÍTICO:** Feature engineering é iterativa e criativa, mas cada *feature* nova deve pagar seu lugar em desempenho medido — não em plausibilidade narrativa.

Eu olhava para as três colunas que criara com tanto esmero e para o veredicto do teste: quase zero. Foi um golpe no ego, mas um presente para o rigor. As *features* não eram ruins — eram redundantes. O sinal que eu julgava estar "esculpindo" já estava lá, embutido nas medidas originais.

O problema, percebi, não era falta de *features*: era excesso. Com trinta variáveis, muitas correlacionadas, meu modelo poderia se confundir. Era hora de trocar o cinzel pela navalha — a navalha de Occam — e começar a podar.
